# Playground Feast — plan/apply, historical features, materialize, online features (SDK e HTTP)

Pré-requisitos:
- `notebooks/01_generate_mock_data.py` já executado (Delta table em `data/offline_store/driver_stats`).
- Redis + RedisInsight no ar (`docker compose up -d` na raiz do repo).
- Para a última célula (chamada HTTP), o feature server precisa estar rodando
  em outro terminal: `uv run feast --chdir feature_repo serve` (porta 6566).

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from deltalake import DeltaTable
from feast import FeatureStore

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
FEATURE_REPO_DIR = REPO_ROOT / "feature_repo"
DELTA_TABLE_PATH = REPO_ROOT / "data" / "offline_store" / "driver_stats"

sys.path.insert(0, str(FEATURE_REPO_DIR))

## Passo 1 — `feast plan` (via CLI)

Nota: `FeatureStore.plan()` no SDK puro exige montar manualmente um objeto
`RepoContents` (o mesmo trabalho que o parser da CLI já faz). Para este
playground, reproduzimos o `feast plan` chamando a CLI via `subprocess`,
que é a forma pública e estável de obter o mesmo resultado.

In [ ]:
result = subprocess.run(
    ["uv", "run", "feast", "--chdir", str(FEATURE_REPO_DIR), "plan"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## Passo 2 — `apply` via SDK

In [ ]:
from entities import driver  # noqa: E402
from feature_services import driver_activity_v1  # noqa: E402
from features import driver_stats_fv  # noqa: E402

store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))
store.apply([driver, driver_stats_fv, driver_activity_v1])
print("apply concluído")

## Passo 3 — Historical features (point-in-time join)

Os timestamps do `entity_df` são derivados dos próprios dados mockados
(min/max de `event_timestamp` por driver), para o notebook continuar
funcionando mesmo que `01_generate_mock_data.py` seja re-executado em
outro momento. Incluímos de propósito um timestamp **antes** do primeiro
evento do driver 1009, para mostrar features nulas quando não há histórico
ainda no ponto no tempo consultado.

In [ ]:
raw = DeltaTable(str(DELTA_TABLE_PATH)).to_pandas().sort_values(["driver_id", "event_timestamp"])

driver_1000_events = raw[raw["driver_id"] == 1000]["event_timestamp"]
driver_1005_events = raw[raw["driver_id"] == 1005]["event_timestamp"]
driver_1009_events = raw[raw["driver_id"] == 1009]["event_timestamp"]

entity_df = pd.DataFrame(
    {
        "driver_id": [1000, 1005, 1009, 1000],
        "event_timestamp": [
            driver_1000_events.iloc[len(driver_1000_events) // 2],  # meio do histórico do driver 1000
            driver_1005_events.iloc[-1],  # último evento conhecido do driver 1005
            driver_1009_events.iloc[0] - pd.Timedelta(hours=1),  # antes de qualquer evento -> features nulas
            datetime.now(timezone.utc),  # "agora" -> deve trazer o último valor materializado do driver 1000
        ],
    }
)

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_stats:latitude",
        "driver_stats:longitude",
        "driver_stats:status",
    ],
).to_df()
training_df

## Passo 4 — Materialize para o Redis

A primeira materialização completa (cobrindo todo o histórico mockado) foi
feita via CLI em `docs/tasks/07-materialize.md`
(`feast materialize <inicio> <fim>`), necessária porque `materialize_incremental`
usa `(agora - ttl)` como início apenas quando não há checkpoint prévio.
Aqui usamos `materialize_incremental`, o padrão do dia a dia: materializa
somente o que é novo desde a última materialização registrada.

In [ ]:
store.materialize_incremental(end_date=datetime.now(timezone.utc))

## Passo 5 — Online features via SDK

In [ ]:
online_sdk = store.get_online_features(
    features=[
        "driver_stats:latitude",
        "driver_stats:longitude",
        "driver_stats:status",
    ],
    entity_rows=[{"driver_id": 1000}, {"driver_id": 1005}],
).to_dict()
online_sdk

## Passo 6 — Online features via HTTP (feature server)

Pré-requisito: em outro terminal, rodar
`uv run feast --chdir feature_repo serve` (porta padrão 6566).

In [ ]:
try:
    resp = requests.post(
        "http://localhost:6566/get-online-features",
        json={
            "features": [
                "driver_stats:latitude",
                "driver_stats:longitude",
                "driver_stats:status",
            ],
            "entities": {"driver_id": [1000, 1005]},
        },
        timeout=15,  # a primeira chamada pode ser mais lenta (warm-up do servidor)
    )
    resp.raise_for_status()
    online_http = resp.json()
    print(online_http)
except requests.exceptions.ConnectionError:
    online_http = None
    print(
        "Feature server não respondeu em localhost:6566 — rode "
        "`uv run feast --chdir feature_repo serve` em outro terminal e "
        "execute esta célula novamente."
    )

## Passo 7 — Comparação SDK vs HTTP

Confere que os valores retornados pelas duas vias batem (mesma fonte no Redis).

In [ ]:
if online_http is not None:
    print("SDK :", online_sdk["latitude"], online_sdk["longitude"], online_sdk["status"])
    print("HTTP:", online_http["results"])